# 호기별 발전 패턴 (2024년 7월)
전처리된 데이터(`solar_hourly_long.csv`, 호기 단위) 기반 — `plan/main/data_strategy.md` §2 화이트리스트
- 사용 가능 **12 호기 × 9 사이트**
- 호기가 최소 단위. 사이트 내 사용/미사용 혼재였던 것(삼천포 #4 등)은 전처리에서 이미 걸러짐.
- ESS 왜곡/일합계만 제공 호기도 전처리 단계에서 제외.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv('../../data/processed/solar_hourly_long.csv', parse_dates=['date', 'datetime'])
df['unit'] = df['unit'].astype(str)
n_units = df[['site','unit']].drop_duplicates().shape[0]
print(f'전체: {len(df):,}행, 호기: {n_units}개 ({df["site"].nunique()} 사이트)')
print(f'기간: {df["date"].min().date()} ~ {df["date"].max().date()}')

df7 = df[(df['date'] >= '2024-07-01') & (df['date'] <= '2024-07-31')].copy()
print(f'\n7월: {len(df7):,}행')

## 1. 사용 가능 12 호기 확정 리스트

`plan/main/data_strategy.md §2`에 정의된 호기 화이트리스트 기반. 데이터 피크 시각 ≤ 15시 기준으로 ESS 왜곡 제거 후 확정. 이 12 호기가 모델 학습 대상.

In [ ]:
# 12 호기 리스트 + 사이트 총용량
UNIT_CAPACITY_KW = {
    '경상대태양광#1': 905,
    '고흥만 수상태양광#1': 63481,
    '광양항세방태양광#1': 2993,
    '구미태양광#1': 992,
    '두산엔진MG태양광#1': 77,
    '삼천포태양광#1': 1097,
    '삼천포태양광#2': 1097,
    '삼천포태양광#3': 1097,
    '영흥태양광#1': 500,
    '영흥태양광#2': 500,
    '영흥태양광#5#1': 3500,
    '예천태양광#1': 2000,
}
summary = pd.DataFrame([{'site_unit': k, 'capacity_kW': v} for k,v in UNIT_CAPACITY_KW.items()])
summary = summary.sort_values('capacity_kW', ascending=False).reset_index(drop=True)
total = summary['capacity_kW'].sum()
summary['share_%'] = (summary['capacity_kW'] / total * 100).round(1)
print(f'총 {len(summary)} 호기, 전체 용량 {total/1000:.2f} MW')
print(summary.to_string(index=False))

## 2. 포트폴리오 전체 — 48개월(2022~2025) 발전 패턴

In [ ]:
# 전체 기간 일별 + 월별 패턴
import matplotlib.dates as mdates

portfolio_daily = df.groupby('date')['gen_kwh'].sum() / 1000  # MWh
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
monthly = df.groupby(['year','month'])['gen_kwh'].sum().reset_index()
monthly['ym'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly['daily_avg_mwh'] = monthly['gen_kwh'] / 1000 / 30  # 월 내 일평균

fig, axes = plt.subplots(2, 1, figsize=(18, 9))

# (상) 전체 기간 일별 총발전량
ax = axes[0]
ax.plot(portfolio_daily.index, portfolio_daily.values, color='#1976D2', linewidth=0.5, alpha=0.7)
# 30일 이동평균
rolling = portfolio_daily.rolling(30, center=True).mean()
ax.plot(rolling.index, rolling.values, color='#E53935', linewidth=2, label='30일 이동평균')
ax.set_ylabel('일 총발전량 (MWh)')
ax.set_title(f'전 기간 일별 포트폴리오 발전량 ({df["date"].min().date()} ~ {df["date"].max().date()})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# (하) 월별 바차트
ax = axes[1]
colors = ['#4CAF50' if m in [3,4,5] else '#FF9800' if m in [6,7,8] else '#F44336' if m in [9,10,11] else '#2196F3' for m in monthly['month']]
ax.bar(monthly['ym'], monthly['daily_avg_mwh'], color=colors, alpha=0.8, width=25)
ax.set_ylabel('월별 일평균 발전량 (MWh)')
ax.set_title('월별 일평균 발전량 (색: 봄=초록, 여름=주황, 가을=빨강, 겨울=파랑)')
ax.grid(True, alpha=0.3, axis='y')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.show()

print(f'전체 기간 평균 일발전량: {portfolio_daily.mean():.0f} MWh')
print(f'최대일: {portfolio_daily.idxmax().date()} ({portfolio_daily.max():.0f} MWh)')
print(f'최저일: {portfolio_daily.idxmin().date()} ({portfolio_daily.min():.0f} MWh)')

## 3. 호기별 규모 비교 — 고흥만 dominance 확인

In [ ]:
# 48개월 호기별 일평균 (원본 MWh — 규모 dominance 시각화 목적)
unit_daily_avg = df.groupby(['site','unit'])['gen_kwh'].sum() / 1000 / ((df['date'].max() - df['date'].min()).days + 1)
unit_daily_avg = unit_daily_avg.sort_values()
labels = [f'{s} #{u}' for s,u in unit_daily_avg.index]

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#E53935' if v > 50 else '#FF7043' if v > 5 else '#FFA726' if v > 1 else '#FFCC80'
          for v in unit_daily_avg.values]
bars = ax.barh(labels, unit_daily_avg.values, color=colors)
for bar, val in zip(bars, unit_daily_avg.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontsize=9)

ax.set_xlabel('일평균 발전량 (MWh, 48개월 평균)')
ax.set_title('호기별 일평균 발전량 — 고흥만 dominance 확인 (원본 MWh, 정규화 X)')
ax.grid(True, alpha=0.3, axis='x')

total = unit_daily_avg.sum()
goheung = unit_daily_avg.get(('고흥만 수상태양광','1'), 0)
ax.text(0.98, 0.02, f'고흥만 비중: {goheung/total*100:.0f}%
전체 일평균: {total:.0f} MWh',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()